In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
import time
import os
import json

    

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using device: {DEVICE}')

In [ ]:
CLEANED_CSV = r'C:\Users\maxle\UCInsure\src\hurricane_model\hurricane_training_data.csv'
CHECKPOINT = r'C:\Users\maxle\UCInsure\src\hurricane_model\hurricane_model.pt'

HURRICANE_COLS = [
    'HRCN_EVNTS',
    'HRCN_AFREQ',
    'HRCN_EXP_AREA',
    'HRCN_EXPB',
    'HRCN_EXPP',
    'HRCN_HLRB',
    'HRCN_HLRP',
    'BUILDVALUE',
    'POPULATION',
    'AREA',
    'SOVI_SCORE',
    'RESL_SCORE',
    'POP_DENSITY',       # engineered in Step 1
    'EXPOSURE_RATIO',    # engineered in Step 1
]

LABEL_COL = 'LABEL_NORM'

Hyper params stored here for now

In [ ]:
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-3
TRAIN_SPLIT = 0.8
RANDOM_SEED = 99
EARLY_STOP = 10

READ CSV

In [ ]:
df = pd.read_csv(CLEANED_CSV)
print(f'loaded {len(df)} rows and {len(df.columns)} columns from {CLEANED_CSV}')

check for na values (SHOULD BE REMOVED)

In [ ]:
missing_values = df.isnull().sum()
print(f'missing values per column:\n{missing_values}')

In [ ]:
X = df[HURRICANE_COLS].values
y = df[LABEL_COL].values

feat_min = X.min(axis=0)
feat_max = X.max(axis=0)

feature_range = feat_max - feat_min
feature_range[feature_range == 0] = 1

X_norm = (X - feat_min) / feature_range
print(f'normalized features shape: {X_norm.shape}')

In [ ]:
scaler_path = CHECKPOINT.replace('.pt', '_scaler.json')
scaler_info = {
    'feature_cols': HURRICANE_COLS,
    'feat_min': feat_min.tolist(),
    'feat_max': feat_max.tolist(),
}
with open(scaler_path, 'w') as f:
    json.dump(scaler_info, f)
print(f'saved feature scaler info to {scaler_path}')
print(f'Feature matrix shape: {X_norm.shape}, Label vector shape: {y.shape}')

In [ ]:
class HurricaneDataset(Dataset):
    def __init__(self,x , y)
        self.X = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
fulldataset = HurricaneDataset(X_norm, y)

train_size = int(TRAIN_SPLIT * len(fulldataset))
val_size = len(fulldataset) - train_size

train_dataset, val_dataset = random_split(fulldataset, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(RANDOM_SEED)
)   

print(f'Train dataset size: {len(train_dataset)}, Validation dataset size: {len(val_dataset)}')

In [ ]:
class HurricaneModel(nn.Module):
    def __init__(self, input_dim, dropoutrate=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropoutrate),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropoutrate),
            
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.network(x)

model = HurricaneModel(input_dim=X_norm.shape[1], dropoutrate=0.3).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')